In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hola


In [ ]:
class RawSignal:
    """
    Clase para manejar señales fisiológicas en formato NumPy.
    Este constructor permite inicializar el objeto 'RawSignal' a partir de un array de datos ,
    con información adicional de los canales y el índice de la primera muestra.
    """
    
    def __init__(self, data, sfreq, info=None, anotaciones=None, first_samp=0):
        """
        Inicializa una instancia de la clase RawSignal.
        
        Parameters
        ----------
        data : np.ndarray
            Matriz de datos con forma '(n_canales , n_muestras)'.
        sfreq : float
            Frecuencia de muestreo de la señal en Hz.
        info: Objeto del tipo  info. Opcional. Por defecto es None.
            Información adicional sobre la señal. El diccionario contiene info relevante de la señal
        anotaciones : Anotaciones
            Objeto de tipo Anotaciones que almacena eventos asociados a la señal y al experimento.
        first_samp : int, optional
            Índice del primer muestreo a utilizar (default es 0).
        
        Raises
        ------
        ValueError
            Si el array 'data' no tiene la forma '(n_canales , n_muestras)'.
        ValueError
            Si el índice 'first_samp' está fuera del rango de la señal.
        """
        # Validar tipo de data
        if not isinstance(data, np.ndarray):
            raise ValueError("El parámetro 'data' debe ser un array de NumPy (np.ndarray).")

        if data.ndim != 2:
            raise ValueError("El array 'data' debe tener dos dimensiones: (n_canales, n_muestras).")

        n_muestras = data.shape[1]
        if not (0 <= first_samp < n_muestras):
            raise ValueError("El índice 'first_samp' está fuera del rango de muestras disponibles.")

        # Asignación de atributos
        self.data = data
        self.sfreq = sfreq
        self.info = info
        self.anotaciones = anotaciones
        self.first_samp = first_samp
        
    def get_data(self, picks=None, start=0, stop=0, reject=None, times=False):
        """
        Obtiene muestras de la señal en un rango dado.

        Parameters
        ----------
        picks : str o array_like, optional
            Canales o índices a extraer. Si es 'None', se seleccionan todos los canales.
        start : float, optional
            Tiempo inicial (en segundos) para extraer muestras (por defecto 0).
        stop : float, optional
            Tiempo final (en segundos) para extraer muestras (por defecto 0, que significa hasta el final de la señal).
        reject : float, optional
            Valor pico a pico de umbral para rechazar canales. Si una muestra supera este umbral, el canal se descarta (por defecto 'None').
        times : bool, optional
            Si es 'True', se retorna también el vector de tiempos asociado a las muestras.

        Returns
        -------
        np.ndarray
            Matriz con los datos seleccionados (n_canales x n_muestras).
        np.ndarray (opcional)
            Vector de tiempos (solo si 'times=True').

        Raises
        ------
        ValueError
            Si los índices seleccionados están fuera de rango.
        """
        
        n_canales, n_muestras = self.data.shape
    
        # Convertir start y stop de segundos a índices de muestra
        start_idx = int(start * self.sfreq)
        stop_idx = int(stop * self.sfreq) if stop > 0 else n_muestras

        if not (0 <= start_idx < stop_idx <= n_muestras):
            raise ValueError("Índices de tiempo fuera de rango.")

        # Seleccionar canales
        if picks is None:
            canales_idx = np.arange(n_canales)
        elif isinstance(picks, (list, np.ndarray)):
            canales_idx = np.array(picks)
        elif isinstance(picks, str):
            # Si info está disponible, buscar canal por nombre
            if self.info and "canales" in self.info:
                canales_idx = [i for i, ch in enumerate(self.info["canales"]) if ch == picks]
                if not canales_idx:
                    raise ValueError(f"Canal '{picks}' no encontrado.")
            else:
                raise ValueError("No se puede buscar por nombre sin metadatos de canales.")
        else:
            raise ValueError("Formato de 'picks' inválido. Debe ser None, str o lista de índices.")

        # Extraer datos seleccionados
        datos = self.data[np.array(canales_idx), start_idx:stop_idx]

        # Aplicar umbral de rechazo si se especifica
        if reject is not None:
            p2p = np.ptp(datos, axis=1)  # pico a pico por canal
            mask = p2p < reject
            datos = datos[mask]

        if times:
            tiempo_vector = np.arange(start_idx, stop_idx) / self.sfreq
            return datos, tiempo_vector

        return datos
    
    def drop_channels(self, ch_names) -> "RawSignal":
        """
        Elimina uno o más canales a partir de ch_names
        Parameters
        ----------
        ch_names:array_like
        Nombres de canales a eliminar
        Returns
        ----------
        RawSignal
        """

        # Validaciones
        if self.info is None or "canales" not in self.info:
            raise ValueError("No se puede eliminar canales sin info['canales'].")

        if isinstance(ch_names, str):
            ch_names = [ch_names]
        elif not isinstance(ch_names, (list, np.ndarray)):
            raise ValueError("'ch_names' debe ser una lista, array o string.")

        # Lista original de canales
        canales_actuales = self.info["canales"]

        # Verificar que todos los canales existan
        for ch in ch_names:
            if ch not in canales_actuales:
                raise ValueError(f"El canal '{ch}' no existe en info['canales'].")

        # Crear máscara para mantener los canales no eliminados
        canales_a_mantener = [i for i, nombre in enumerate(canales_actuales) if nombre not in ch_names]

        # Filtrar data y actualizar info
        nueva_data = self.data[canales_a_mantener, :]
        nueva_info = self.info.copy()
        nueva_info["canales"] = [canales_actuales[i] for i in canales_a_mantener]

        # Crear y devolver nueva instancia de RawSignal
        return RawSignal(
            data=nueva_data,
            sfreq=self.sfreq,
            info=nueva_info,
            anotaciones=self.anotaciones,
            first_samp=self.first_samp
        )